In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.day07;
CREATE VOLUME IF NOT EXISTS workspace.day07.raw;  

RED LINE ON 2ND LINE day07 because day07 is not present it is creting now only thats why it is not recognizing the name

In [0]:
import json, random
from datetime import date, timedelta
 
VOL = "/Volumes/workspace/day07/raw" 
random.seed(6)          # fixed seed -> the whole class gets identical numbers today
 
cities = ["Hyderabad", "Bengaluru", "Chennai", "Pune", "Kolkata"]
items  = ["Rice 5kg", "Toor Dal 1kg", "Sunflower Oil 1L", "Atta 10kg", "Sugar 1kg"]
start  = date(2026, 7, 1)
 
rows = ["order_id,order_date,city,item,quantity,amount"]
for i in range(1, 501):
    dt = start + timedelta(days=random.randint(0, 30))
    rows.append("ORD{:05d},{},{},{},{},{}".format(
        i, dt, random.choice(cities), random.choice(items),
        random.randint(1, 5), round(random.uniform(80, 2400), 2)))
 
# one deliberately broken row: quantity is a word, not a number
rows.append("ORD00501,2026-07-15,Hyderabad,Rice 5kg,two,999.00")
 
with open(VOL + "/sales.csv", "w") as f:
    f.write("\n".join(rows) + "\n")
 
stores = [
  {"city":"Hyderabad","store_id":"HYD01","manager":"R. Padmaja","opened":2019},
  {"city":"Bengaluru","store_id":"BLR01","manager":"S. Iyer","opened":2021},
  {"city":"Chennai",  "store_id":"MAA01","manager":"K. Vetrivel","opened":2018},
  {"city":"Pune",     "store_id":"PNQ01","manager":"A. Deshmukh","opened":2022},
  {"city":"Kolkata",  "store_id":"CCU01","manager":"D. Ghosh","opened":2020}]
 
with open(VOL + "/stores.json", "w") as f:        # line-delimited (Spark default)
    for s in stores:
        f.write(json.dumps(s) + "\n")
 
with open(VOL + "/stores_pretty.json", "w") as f: # pretty-printed (needs multiLine)
    json.dump(stores, f, indent=2)
 
print("wrote", len(rows) - 1, "data rows to", VOL)

wrote 501 data rows to /Volumes/workspace/day07/raw


In [0]:
PATH = VOL + "/sales.csv"
df_naive = spark.read.csv(PATH)
df_naive.printSchema()
df_naive.show(3, truncate = False)

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)

+--------+----------+---------+------------+--------+-------+
|_c0     |_c1       |_c2      |_c3         |_c4     |_c5    |
+--------+----------+---------+------------+--------+-------+
|order_id|order_date|city     |item        |quantity|amount |
|ORD00001|2026-07-26|Kolkata  |Rice 5kg    |4       |1848.34|
|ORD00002|2026-07-02|Hyderabad|Toor Dal 1kg|5       |1170.99|
+--------+----------+---------+------------+--------+-------+
only showing top 3 rows


In [0]:
df_infer = (spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(PATH)
)
df_infer.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- item: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- amount: double (nullable = true)



In [0]:
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, DoubleType, DateType)

sales_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("city", StringType(), True),
    StructField("item", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("amount", DoubleType(), True),
])
def read_sales(mode):
    return (spark.read
            .schema(sales_schema)
            .option("header", "true")
            .option("mode", mode)
            .csv(PATH)
            )

In [0]:
df_perm = read_sales("PERMISSIVE") # the default
df_perm.printSchema()
print("PERMISSIVE rows:", df_perm.count())
df_perm.filter("quantity IS NULL").show()


root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- item: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)

PERMISSIVE rows: 501
+--------+----------+---------+--------+--------+------+
|order_id|order_date|     city|    item|quantity|amount|
+--------+----------+---------+--------+--------+------+
|ORD00501|2026-07-15|Hyderabad|Rice 5kg|    NULL| 999.0|
+--------+----------+---------+--------+--------+------+



In [0]:
print("DROPMALFORMED rows:", read_sales("DROPMALFORMED").count())
try:
    read_sales("FAILFAST").count()
except Exception as e:
    print("FAILFAST:", type(e).__name__)

DROPMALFORMED rows: 501


In [0]:
spark.read.json(VOL + "/stores.json").show()

spark.read.json(VOL + "/stores_pretty.json").printSchema()

(spark.read.option("multiLine", "true")
           .json(VOL + "/stores_pretty.json")).show()

+---------+-----------+------+--------+
|     city|    manager|opened|store_id|
+---------+-----------+------+--------+
|Hyderabad| R. Padmaja|  2019|   HYD01|
|Bengaluru|    S. Iyer|  2021|   BLR01|
|  Chennai|K. Vetrivel|  2018|   MAA01|
|     Pune|A. Deshmukh|  2022|   PNQ01|
|  Kolkata|   D. Ghosh|  2020|   CCU01|
+---------+-----------+------+--------+

root
 |-- _corrupt_record: string (nullable = true)

+---------+-----------+------+--------+
|     city|    manager|opened|store_id|
+---------+-----------+------+--------+
|Hyderabad| R. Padmaja|  2019|   HYD01|
|Bengaluru|    S. Iyer|  2021|   BLR01|
|  Chennai|K. Vetrivel|  2018|   MAA01|
|     Pune|A. Deshmukh|  2022|   PNQ01|
|  Kolkata|   D. Ghosh|  2020|   CCU01|
+---------+-----------+------+--------+



In [0]:
OUT = VOL + "/curated/sales_parquet"
df_clean = read_sales("DROPMALFORMED")
df_clean.write.mode("overwrite").parquet(OUT)
display(dbutils.fs.ls(OUT))

path,name,size,modificationTime
dbfs:/Volumes/workspace/day07/raw/curated/sales_parquet/_SUCCESS,_SUCCESS,0,1786458808000
dbfs:/Volumes/workspace/day07/raw/curated/sales_parquet/_committed_1519655307988701572,_committed_1519655307988701572,124,1786375345000
dbfs:/Volumes/workspace/day07/raw/curated/sales_parquet/_committed_8730367670476405345,_committed_8730367670476405345,234,1786458808000
dbfs:/Volumes/workspace/day07/raw/curated/sales_parquet/_committed_vacuum4896840501317760224,_committed_vacuum4896840501317760224,96,1786458809000
dbfs:/Volumes/workspace/day07/raw/curated/sales_parquet/_started_8730367670476405345,_started_8730367670476405345,0,1786458808000
dbfs:/Volumes/workspace/day07/raw/curated/sales_parquet/part-00000-tid-8730367670476405345-e6b433a7-6a11-41c3-8264-8113718048e4-144-1.c000.snappy.parquet,part-00000-tid-8730367670476405345-e6b433a7-6a11-41c3-8264-8113718048e4-144-1.c000.snappy.parquet,7723,1786458808000


In [0]:
df_pq = spark.read.parquet(OUT)
df_pq.printSchema()
display(df_pq)

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- item: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)



order_id,order_date,city,item,quantity,amount
ORD00001,2026-07-26,Kolkata,Rice 5kg,4,1848.34
ORD00002,2026-07-02,Hyderabad,Toor Dal 1kg,5,1170.99
ORD00003,2026-07-25,Chennai,Sunflower Oil 1L,1,712.66
ORD00004,2026-07-26,Bengaluru,Atta 10kg,5,1331.06
ORD00005,2026-07-04,Bengaluru,Sugar 1kg,5,1702.9
ORD00006,2026-07-24,Chennai,Sugar 1kg,1,2038.73
ORD00007,2026-07-11,Hyderabad,Sunflower Oil 1L,4,1947.09
ORD00008,2026-07-15,Hyderabad,Toor Dal 1kg,3,306.32
ORD00009,2026-07-02,Kolkata,Toor Dal 1kg,3,1209.15
ORD00010,2026-07-27,Bengaluru,Sugar 1kg,5,1575.41


In [0]:
print("rows read backs:", df_pq.count())

rows read backs: 500


In [0]:
print("DROP MALFORMED rows:", read_sales("DROPMALFORMED").count())

#print("DROPMALFORMED rows with collect:", read_sales("DROPMALFORMED").collect())

print("collect() ->", len(read_sales("DROPMALFORMED").collect()))

DROP MALFORMED rows: 501
collect() -> 500


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6554359424739649>, line 1
----> 1 df = read_sales("DROPMALFORMED")
      3 print("count:", df.count())
      4 print("collect length:", len(df.collect()))

NameError: name 'read_sales' is not defined